In [ ]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()


# 오류 발생시
# sklearn.set_config(display="text")

In [2]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

output_dir = "/Users/kdt_0900_cjs/ml/resource/dataset"
os.makedirs(output_dir, exist_ok=True)
# https://www.kaggle.com/datasets/sanikamal/horses-or-humans-dataset
api.dataset_download_files("drgfreeman/rockpaperscissors", path=output_dir, unzip=True)

print("다운로드 완료")

Dataset URL: https://www.kaggle.com/datasets/drgfreeman/rockpaperscissors
다운로드 완료


In [5]:
base_path = "/Users/kdt_0900_cjs/ml/resource/dataset"
source = os.path.join(base_path,"rps-cv-images")
destination = s.path.join(base_path,"rps-cv-images")
classes = ("paper", "rock", "scissors")

images_class = {}

for label in classes:
    folder = source / label

    if not folder.is_dir():
        raise FileNotFoundError(f"원본 폴더 없음: {folder}")

    images = sorted(folder.glob("*.png"))

    if not images:
        raise ValueError(f"이미지 없음: {folder}")

    images_class[label] = images

    
rng = random.Random(42)

for label, images in images_class.items():
    rng.shuffle(images)

    total = len(images)
    training_count = int(total * 0.8)
    test_count = int(total * 0.1)

    splits = {
        "training": images[:training_count],
        "test": images[training_count:training_count + test_count],
        "validation": images[training_count + test_count:],
    }

    for split, files in splits.items():
        target = destination / split / label
        target.mkdir(parents=True, exist_ok=True)

    for file in files:
        shutil.copy2(file, target / file.name)

    print(f"{split}/{label}: {len(files)}장")

NameError: name 'os' is not defined

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


target_transfrom = lambda y: torch.tensor([y], dtype=torch.float32)

In [ ]:
base_path = output_dir
train_path = os.path.join(base_path, "horse-or-human", "train")
test_path = os.path.join(base_path, "horse-or-human", "validation")


train_dataset = ImageFolder(train_path, transform=train_transform, target_transform = target_transform)
test_dataset = ImageFolder(test_path, transform=test_transform, target_transform=target_transform)